<a href="https://colab.research.google.com/github/emmakelly7/ds2002-fa26/blob/main/2026_09_11_%E2%80%94_SQL_Challenge_Set_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · SQL Challenge Set

**Lab — 2026-09-11 · Fall 2026**  

---

## Lab 03 — SQL Challenge Set

Seven questions, one query each. Every query has to produce the right answer when the notebook is run from a fresh kernel, top to bottom.

Two rules that matter as much as getting the answer:

- **Check the row count** against what you expect before you believe a result.
- **Decide what to do about the untagged track and the unplayed tracks.** Several of these questions have a defensible answer either way; what is not defensible is not noticing they exist.

In [29]:
import sqlite3, pandas as pd
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER, genre TEXT, seconds INTEGER);
CREATE TABLE plays (play_id INTEGER PRIMARY KEY, track_id INTEGER, user TEXT, played_on TEXT);
INSERT INTO artists VALUES
 (1,'Nova Waves','US'),(2,'The Blue Ridge','US'),(3,'Kestrel','UK'),(4,'Marisol','ES');
INSERT INTO tracks VALUES
 (10,'Skyline',1,'Pop',201),(11,'Undertow',1,'Pop',240),(12,'Foothills',2,'Folk',185),
 (13,'Aurora',3,'Electronic',300),(14,'Nightfall',3,'Electronic',275),(15,'Sol',4,'Latin',210),
 (16,'Coastline',2,'Folk',199),(17,'Ridgeline',2,'Folk',225),(18,'Untitled Demo',3,NULL,150);
INSERT INTO plays VALUES
 (100,10,'ava','2026-09-01'),(101,10,'ben','2026-09-01'),(102,13,'ava','2026-09-02'),
 (103,13,'cara','2026-09-02'),(104,14,'ben','2026-09-03'),(105,12,'ava','2026-09-03'),
 (106,15,'dan','2026-09-04'),(107,10,'cara','2026-09-04'),(108,13,'dan','2026-09-05'),
 (109,16,'ava','2026-09-05'),(110,11,'ben','2026-09-06');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)
print('ready')

ready


### Q1 — Every track with its artist's name and country.

*Expected: 9 rows, one per track.*

In [30]:
q1 = q('''
SELECT tracks.title, artists.name, artists.country
FROM tracks
JOIN artists
ON tracks.artist_id = artists.artist_id
''')

q1

,title,name,country
0,Skyline,Nova Waves,US
1,Undertow,Nova Waves,US
2,Foothills,The Blue Ridge,US
3,Aurora,Kestrel,UK
4,Nightfall,Kestrel,UK
5,Sol,Marisol,ES
6,Coastline,The Blue Ridge,US
7,Ridgeline,The Blue Ridge,US
8,Untitled Demo,Kestrel,UK


The q() makes the query run on the database and then it outputs a table. The SELECT function allows it to look at certain columns, which I listed, and then I listed FROM tracks to say where to get it. JOIN artists combines the two tables, and then ON tracks.artist_id = artists.artist_id combines the track and artist when artist_id for tracks and artists is the same.

### Q2 — Which genre has the longest average track length?

Return the genre and the average, not just the name.

In [31]:
q2 = q('''
SELECT genre, AVG(seconds)
FROM tracks
GROUP BY genre
ORDER BY AVG(seconds) DESC
LIMIT 1
''')

q2

,genre,AVG(seconds)
0,Electronic,287.5


Using SELECT again, I looked at the genre and calculated the average number of seconds for each of the tracks in the genre, and then put the tracks under the same genre into the same group and ordered them by their average number of seconds and sorted them from greatest to least in descending order, and then asked it to output only the first result.

### Q3 — For each user: how many plays, and how many distinct tracks?

Someone who played one track four times is a different listener from someone who played four different tracks. Your result should make that visible.

In [32]:
q3 = q('''
SELECT user, COUNT(play_id) AS plays, COUNT(DISTINCT track_id) AS tracks
FROM plays
GROUP BY user
''')

q3

,user,plays,tracks
0,ava,4,4
1,ben,3,3
2,cara,2,2
3,dan,2,2


I asked it to count each play and then count the different tracks that were listened to, and then group it by person/user. It shows that Ava had four plays of four tracks, Ben listened to three, and Cara and Dan both listened to two. No one repeated a track.

### Q4 — Which tracks have never been played?

*Expected: 2 rows.* Hint: `LEFT JOIN` and then keep the rows where the right side came back `NULL`.

In [33]:
q4 = q('''
SELECT tracks.track_id, tracks.title
FROM tracks
LEFT JOIN plays
ON tracks.track_id = plays.track_id
WHERE plays.track_id IS NULL
''')

q4

,track_id,title
0,17,Ridgeline
1,18,Untitled Demo


LEFT JOIN looks at every track even if there isn't a listing under the plays, and then for tracks WHERE play IS NULL, it finds the tracks that have not been played and outputs two rows.

### Q5 — Rank artists by total listening time.

Sum the seconds actually listened across all plays, most to least, and include a minutes column rounded to one decimal.

In [34]:
q5 = q('''
SELECT artists.name, SUM(tracks.seconds) AS seconds, ROUND(SUM(tracks.seconds) / 60.0, 1) AS minutes
FROM artists
JOIN tracks
ON artists.artist_id = tracks.artist_id
JOIN plays
ON tracks.track_id = plays.track_id
GROUP BY artists.name
ORDER BY seconds DESC
''')

q5

,name,seconds,minutes
0,Kestrel,1175,19.6
1,Nova Waves,843,14.1
2,The Blue Ridge,384,6.4
3,Marisol,210,3.5


I joined tracks, plays, and artists, so each artist was with their track. Then for seconds actually listened across all plays, I used SUM(tracks.seconds) to add the track length for each play and then used 60 to convert the seconds to minutes and then rounded it to 1 decimal place. Then, I used ORDER BY seconds DESC to order it in descending order.

### Q6 — Which tracks are missing a genre?

Return the track id and title. Then, in a comment, say what `WHERE genre != 'Pop'` would have done to these rows and why.

In [35]:
q6 = q('''
SELECT track_id, title
FROM tracks
WHERE genre is NULL
''')

q6

,track_id,title
0,18,Untitled Demo


To find the tracks missing from the genre, I searched with the track_id and title under tracks and then looked where the genre was listed as null. It output one track that was Untitled. If I had used WHERE genre != 'Pop', it would have listed multiple tracks and not the Untitled Demo because it is just asking which tracks were not listed under Pop, not which were missing a genre in general. Therefore, you need to use is NULL to get the correct result.

### Q7 — Plays per day.

`played_on` is stored as text like `'2026-09-01'`. Count plays per date, earliest first, and include the number of distinct users active that day.

In [36]:
q7 = q('''
SELECT played_on, COUNT(play_id) AS plays, COUNT(DISTINCT user) AS users
FROM plays
GROUP BY played_on
ORDER BY played_on
''')

q7

,played_on,plays,users
0,2026-09-01,2,2
1,2026-09-02,2,2
2,2026-09-03,2,2
3,2026-09-04,2,2
4,2026-09-05,2,2
5,2026-09-06,1,1


SELECT played_on looks at date, and then COUNT looks at the amount of plays per day, and the. COUNT(DISTINCT user) AS users looks at the number of different users that day. GROUP BY played_on puts the plays from the same day together, and then ORDER lists them in date order.

### Validate your work

**TODO:** uncomment these and make them pass. Assign your query results to the variables as you go — for example `q1 = q('''...''')`.

In [37]:
assert len(q1) == 9, 'Q1 should return one row per track'
assert len(q4) == 2, 'Q4: two tracks have never been played'
assert q3['plays'].sum() == 11, 'Q3 should account for all 11 plays'
print('checks passed.')

checks passed.


### Write-up

Pick the query that gave you the most trouble and explain what you had wrong before you had it right. Name the specific misunderstanding — "I put the aggregate in WHERE" or "I used an inner join and lost the tracks with no plays" — not "it was confusing."

The query that gave me the most trouble was Question 5. I originally misread the question and only calculated minutes instead of adding total seconds. I also did not fully understand how to find the total listening time. Then, I realized I needed to join the artists, tracks, and play so play connected to length. Then, I used SUM(tracks.seconds) to find the seconds for each artists and converted them to minutes and grouped them by artist. Finally, I ordered them in descending order.